# Step 1a wf_v3.1 Step-3 pilot (3 entities, prompt v2, split pass)
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v3.1`, folder `step1a/`. Jobs: `results/loop1_v31/manifest.json`. Responses are committed back to the same branch.

**Before running:** Runtime > Change runtime type > **A100 (80 GB)**. Colab Secrets `HF_TOKEN` and `GH_TOKEN` with notebook access on.

In [ ]:
#@title 1. Tokens, clone/pull wf-v3.1, git identity
# Needs Colab Secrets: HF_TOKEN and GH_TOKEN, notebook access on.
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
BRANCH = "wf-v3.1"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** `pip install vllm` prints dependency-resolver warnings. Ignore them. Then **Runtime > Restart session**, rerun cell 1, skip cell 2.

In [ ]:
#@title 2. Install vLLM (a few minutes; ignore resolver warnings, then restart the session and skip this cell)
!pip install -q vllm

In [ ]:
#@title 3. GPU check (the 32B AWQ model needs >= 40 GB)
ALLOW_SMALL_GPU = False  #@param {type:"boolean"}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GB")
assert gb >= 40 or ALLOW_SMALL_GPU, (
    f"only {gb:.0f} GB: the 32B model does not fit with a 21K-token prompt. Use an A100, or tick ALLOW_SMALL_GPU "
    "and set MODEL in cell 4 to Qwen/Qwen2.5-14B-Instruct-AWQ")

In [ ]:
#@title 4. Load the model
MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"  #@param ["Qwen/Qwen2.5-32B-Instruct-AWQ", "Qwen/Qwen2.5-14B-Instruct-AWQ"]
REVISION = ""  #@param {type:"string"}
MAX_LEN = 32768
import vllm
from huggingface_hub import model_info
from vllm import LLM, SamplingParams
if not REVISION:
    REVISION = model_info(MODEL).sha  # resolve once, so the cache key names the exact weights
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization="awq",
          max_model_len=MAX_LEN, gpu_memory_utilization=0.92)
tok = llm.get_tokenizer()
MODEL_ID = f"{MODEL}@{REVISION}"  # used in cache keys

In [ ]:
#@title 5. Manifest and exact prompt token counts (stops if any prompt > 30000)
import json, sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from pathlib import Path
from s1a import llm_cache as lc
O = Path("results/loop1_v31")
manifest = json.loads((O / "manifest.json").read_text(encoding="utf-8"))
jobs = manifest["jobs"] if isinstance(manifest, dict) else manifest
def chat_prompt(text):
    return tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
prompts = {j["job"]: (O / j["prompt"]).read_text(encoding="utf-8") for j in jobs}
schemas = {j["job"]: json.loads((O / j["schema"]).read_text(encoding="utf-8")) for j in jobs}
counts = {n: len(tok(chat_prompt(p), add_special_tokens=False).input_ids) for n, p in prompts.items()}
print(len(jobs), "jobs; tokens: min", min(counts.values()), "max", max(counts.values()))
too_long = {n: c for n, c in counts.items() if c > 30000}
assert not too_long, f"prompt over 30000 tokens: {too_long}; lower k (W1) before running"


In [ ]:
#@title 6. Generation helper (temperature 0, guided JSON per job schema, unguided retry)
# vLLM >= 0.11: StructuredOutputsParams(json=...) via `structured_outputs`; older: GuidedDecodingParams via `guided_decoding`.
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)

def generate(text, max_tokens, schema):
    """(raw text, guided, finish_reason). If guided decoding raises for this schema, retry once without guidance."""
    try:
        out = llm.generate([chat_prompt(text)], sampling(max_tokens, schema))[0].outputs[0]
        return out.text, True, out.finish_reason
    except Exception as e:
        print("  guided decoding failed, retrying unguided:", type(e).__name__, str(e)[:200])
        out = llm.generate([chat_prompt(text)], sampling(max_tokens, None))[0].outputs[0]
        return out.text, False, out.finish_reason

In [ ]:
#@title 7. Run all jobs (entity jobs, then split jobs; cached by model+revision, params, prompt, run_tag)
import hashlib, json, time
order = [j for j in jobs if j["kind"] == "entity"] + [j for j in jobs if j["kind"] == "split"]
t_all = time.time()
first_run_s = None
for n, job in enumerate(order, 1):
    name, prompt = job["job"], prompts[job["job"]]
    max_tokens = min(job["max_tokens"], MAX_LEN - counts[name] - 64)
    schema_sha = hashlib.sha256(json.dumps(schemas[name], sort_keys=True).encode("utf-8")).hexdigest()[:16]
    params = {"temperature": 0, "max_tokens": max_tokens, "guided_json": job["schema"], "schema_sha": schema_sha}
    k = lc.key(MODEL_ID, params, prompt, job["run_tag"])
    path = O / "responses" / f"{name}.jsonl"
    row, t0 = lc.get(path, k), time.time()
    cached = row is not None
    if not cached:
        raw, guided, finish = generate(prompt, max_tokens, schemas[name])
        parsed, errors, _ = lc.parse_job(raw, job["format"], schemas[name])
        row = {"prompt_sha": hashlib.sha256(prompt.encode("utf-8")).hexdigest(), "model": MODEL, "revision": REVISION,
               "vllm": vllm.__version__, "params": params, "run_tag": job["run_tag"], "job": name,
               "format": job["format"], "guided": guided, "finish_reason": finish, "raw": raw, "parsed": parsed, "errors": errors}
        lc.put(path, k, row)
        if first_run_s is None:
            first_run_s = time.time() - t0
            print(f"first job took {first_run_s:.0f} s; rough total for {len(order)} jobs: {first_run_s * len(order) / 60:.0f} min")
    n_obj = lc.parse_job(row["raw"], job["format"], schemas[name])[2]
    print(f"[{n}/{len(order)}] {name}: tokens in {counts[name]}, {time.time() - t0:.0f} s{' (cached)' if cached else ''}, "
          f"objects {n_obj}, errors {len(row['errors'])}, guided {row['guided']}, finish {row.get('finish_reason')}", row["errors"][:2])
print(f"elapsed {(time.time() - t_all) / 60:.1f} min")

In [ ]:
#@title 8. Commit and push results/loop1_v31/responses to wf-v3.1
!bash colab/sync.sh push-results "v3.1 pilot responses"